In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import seaborn
import warnings
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_scheduler
from tqdm import tqdm
import torch.nn as nn
import numpy as np

warnings.filterwarnings('ignore')

- Load and Inspect Data

In [5]:
dev_df = pd.read_parquet('dev-00000-of-00001.parquet')
train_df = pd.read_parquet('train-00000-of-00001.parquet')

combined_df = pd.concat([dev_df, train_df], ignore_index=True)
combined_df.head()

,id,text,anger,disgust,fear,joy,sadness,surprise,emotions
0,yor_dev_track_a_00001,"lati ṣami ayẹyẹ ọdun iṣẹṣe, ijọba eko kede ọjọ...",0,0,0,0,0,0,[]
1,yor_dev_track_a_00002,iran plane crash: ẹ wo ìgbà mẹ́jọ tí wọ́n fi à...,0,0,0,0,0,1,[surprise]
2,yor_dev_track_a_00003,"kano rape cases: muhammad, ẹni ọdún mẹjìlélọgb...",0,0,0,0,0,1,[surprise]
3,yor_dev_track_a_00004,àwọn àjẹ́ ń yọ sí mi lójú oorun láti wá dara p...,0,0,0,0,0,0,[]
4,yor_dev_track_a_00005,ogun ritual killing: ọ̀kan lára àwọn afurasí m...,0,0,0,0,0,1,[surprise]


In [6]:
test_df = pd.read_parquet('test-00000-of-00001.parquet')
test_df.head()

,id,text,anger,disgust,fear,joy,sadness,surprise,emotions
0,yor_test_track_a_00001,ẹ fura o! wọn fẹẹ f’ọrọ ẹsin da wahala silẹ ni...,0,0,1,0,0,0,[fear]
1,yor_test_track_a_00002,"ijọba kwara fẹẹ pin ounjẹ faraalu, wọn yoo faw...",0,0,0,1,0,0,[joy]
2,yor_test_track_a_00003,priapism: àìsàn ǹkan ọmọkùnrin tó ń lè gìdìgbà...,0,0,0,0,0,0,[]
3,yor_test_track_a_00004,"plane crash: ọkọ̀ báàlù ṣìnà, ó já wọ ilé onílé",0,0,0,0,1,0,[sadness]
4,yor_test_track_a_00005,ibadan: olóòlù kò ni kọ́ja agboolé rẹ̀ fún ọdú...,0,0,0,0,0,0,[]


In [7]:
print(f"Dimension of the combined dataset: {combined_df.shape}")
print(f"Dimension of the test dataset: {test_df.shape}")

Dimension of the combined dataset: (3986, 9)
Dimension of the test dataset: (3000, 9)


- Check for null or missing values

In [8]:
combined_df['text'].isna().sum()

np.int64(0)

- Check for duplicates to drop them if found.

In [9]:
print(f"Total text rows before dropping duplicates: {len(combined_df['text'])}")
if combined_df['text'].duplicated().any():
  combined_df.drop_duplicates(subset=['text'], inplace=True)
  print(f"Total text rows after dropping duplicates: {len(combined_df['text'])}")

Total text rows before dropping duplicates: 3986
Total text rows after dropping duplicates: 3489


In [10]:
print(f"Test set size: {len(test_df)}")

Test set size: 3000


- Split dataset into training and testing sets

In [11]:
x_train = combined_df['text']
x_test  = test_df['text']

emotion_cols = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']

y_train = combined_df[emotion_cols]
y_test  = test_df[emotion_cols]

In [12]:
len(x_train)

3489

In [13]:
print(y_train)
print(y_test)

      anger  disgust  fear  joy  sadness  surprise
0         0        0     0    0        0         0
1         0        0     0    0        0         1
2         0        0     0    0        0         1
3         0        0     0    0        0         0
4         0        0     0    0        0         1
...     ...      ...   ...  ...      ...       ...
3981      0        0     0    0        0         0
3982      1        0     0    0        0         0
3983      0        0     0    0        0         0
3984      0        0     0    0        0         0
3985      0        0     0    0        0         0

[3489 rows x 6 columns]
      anger  disgust  fear  joy  sadness  surprise
0         0        0     1    0        0         0
1         0        0     0    1        0         0
2         0        0     0    0        0         0
3         0        0     0    0        1         0
4         0        0     0    0        0         0
...     ...      ...   ...  ...      ...       ...
2995  

- Tokenization using AfroXLMR's tokenizer

In [14]:
checkpoint = "Davlan/afro-xlmr-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [15]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=6, problem_type="multi_label_classification")

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float)
        }

train_dataset = EmotionDataset(x_train.tolist(), y_train.values, tokenizer)
test_dataset = EmotionDataset(x_test.tolist(), y_test.values, tokenizer)

- Create DataLoaders

In [17]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

- Set Up Optimizer and Device

In [32]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

optimizer = AdamW(model.parameters(), lr=4e-5)

num_epochs = 8
num_training_steps = num_epochs * len(train_loader)

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)

- Training Loop

In [33]:
pos_counts = torch.tensor(y_train.sum(axis=0).values, dtype=torch.float)
neg_counts = len(y_train) - pos_counts
raw_weight = neg_counts / pos_counts
pos_weight = torch.clamp(raw_weight, max=10.0).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print("Capped weights:", dict(zip(emotion_cols, pos_weight.cpu().numpy().round(2))))

Capped weights: {'anger': np.float32(10.0), 'disgust': np.float32(10.0), 'fear': np.float32(10.0), 'joy': np.float32(10.0), 'sadness': np.float32(2.58), 'surprise': np.float32(10.0)}


In [34]:
model.train()

for epoch in range(num_epochs):
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for batch in loop:
        batch  = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop('labels')

        outputs = model(**batch) # forward pass without internal loss
        loss    = loss_fn(outputs.logits, labels) # weighted loss

        loss.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} avg loss: {total_loss / len(train_loader):.4f}")

Epoch 1: 100%|██████████| 219/219 [01:39<00:00,  2.20it/s, loss=0.0697]


Epoch 1 avg loss: 0.3967


Epoch 2: 100%|██████████| 219/219 [01:38<00:00,  2.21it/s, loss=0.0547]


Epoch 2 avg loss: 0.3153


Epoch 3: 100%|██████████| 219/219 [01:38<00:00,  2.22it/s, loss=0.0912]


...

Epoch 6 avg loss: 0.1198


Epoch 7: 100%|██████████| 219/219 [01:38<00:00,  2.22it/s, loss=0.159]


Epoch 7 avg loss: 0.0952


Epoch 8: 100%|██████████| 219/219 [01:38<00:00,  2.21it/s, loss=0.0481]

Epoch 8 avg loss: 0.0788


- Evaluation

In [43]:
model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)

        logits = outputs.logits
        preds  = torch.sigmoid(logits) > 0.40
        all_preds.append(preds.cpu().numpy())
        all_labels.append(batch['labels'].cpu().numpy())

import numpy as np
all_preds  = np.vstack(all_preds)
all_labels = np.vstack(all_labels)

print(f"\n{classification_report(all_labels, all_preds, target_names=emotion_cols)}")

Evaluating: 100%|██████████| 188/188 [00:26<00:00,  7.08it/s]


              precision    recall  f1-score   support

       anger       0.40      0.39      0.39       196
     disgust       0.30      0.33      0.31        80
        fear       0.39      0.27      0.32        82
         joy       0.37      0.46      0.41       274
     sadness       0.59      0.77      0.66       836
    surprise       0.27      0.31      0.29       256

   micro avg       0.47      0.56      0.51      1724
   macro avg       0.39      0.42      0.40      1724
weighted avg       0.46      0.56      0.50      1724
 samples avg       0.29      0.30      0.29      1724



In [44]:
model.save_pretrained("yoruba_emotion_model")
tokenizer.save_pretrained("yoruba_emotion_tokenizer")
print("Saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved.


In [ ]:
model.eval()
sample_text = x_test.tolist()[0]
encoding = tokenizer(sample_text, return_tensors='pt',
                     truncation=True, max_length=128).to(device)

with torch.no_grad():
    logits = model(**encoding).logits
    probs = torch.sigmoid(logits)

print("Sample text:", sample_text[:80])
print("Probs:", dict(zip(emotion_cols, probs[0].cpu().numpy().round(3))))

Sample text: ẹ fura o! wọn fẹẹ f’ọrọ ẹsin da wahala silẹ nilẹ yoruba – dss pariwo
Probs: {'anger': np.float32(0.414), 'disgust': np.float32(0.212), 'fear': np.float32(0.193), 'joy': np.float32(0.498), 'sadness': np.float32(0.504), 'surprise': np.float32(0.473)}


- Push model and tokenizer to HuggingFace due to size.

In [45]:
from huggingface_hub import notebook_login
notebook_login()

In [46]:
model.push_to_hub("lapppy1/yoruba-emotion-model")
tokenizer.push_to_hub("lapppy1/yoruba-emotion-tokenizer")
print("Done. Model is live on HuggingFace.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...y2uqg0z/model.safetensors:   0%|          | 17.7kB / 1.11GB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpkum9dzh2/tokenizer.json: 100%|##########| 16.8MB / 16.8MB            

Done. Model is live on HuggingFace.
